# Chapter 02-01 · What is a row?

**Label:** Core  |  **Time:** ~45 minutes  |  **Difficulty:** gentle to read, hard to get right in practice

**Prerequisites:** 00-04. Module 01 is optional; if you skipped it you should still be able to
read `groupby` and `merge`.

**Position in the learning path:** module 02 (Data literacy and EDA), chapter 1 of 8. Before:
**00-04**. After: **02-02**, on where data comes from.

---

## Why this matters

Every project has a first question, and it is not "which model?". It is:

> **What does one row represent?**

It sounds trivial. It is the question that decides what your target means, which features are even
possible, how you must split the data, and whether an average you compute is the average anybody
wanted. Get it wrong and everything downstream is technically correct and answers a question nobody
asked.

In this chapter the same twelve rental events are arranged four different ways. Each arrangement is
a valid dataset. They give different averages, support different models, and require different
splits - and choosing between them is a decision *you* make, not something the data tells you.

The failure lab computes a completely reasonable average two completely reasonable ways and gets
2.40 and 2.17. Both are right. Only one of them answers the question that was asked.

## What you will be able to do

By the end of this chapter you can:

1. **State** what one row represents in a dataset, in a sentence, before doing anything else.
2. **Reshape** an event log into different units of observation and say what each one is for.
3. **Classify** every column as target, feature, identifier or metadata.
4. **Name** the measurement scale of a column and say what arithmetic is legitimate on it.
5. **Diagnose** an average computed at the wrong grain, including the mean-of-means trap.

## Warm-up: retrieve, do not reread

From memory:

1. What is a confounder, and what makes something one?
2. Why can a variable be an excellent predictor and a useless lever?
3. What is the one question to ask of every candidate feature?
4. What does a random seed guarantee, and what does it not?

<br>

*Answers: (1) it influences both who gets the treatment and the outcome - both arrows are required.
(2) it can carry information about something else; intervening destroys the information without
changing the cause. (3) will I actually have this value at the moment I make the prediction?
(4) repeatability; not correctness and not representativeness.*

## The situation

Maria's system logs one line every time somebody takes a bike out. Twelve rentals, three days, two
stations, five customers.

Her question sounds simple: **"how busy are we, and how long do people keep the bikes?"**

We are going to answer it four times.

In [ ]:
import numpy as np
import pandas as pd

events = pd.DataFrame({
    "rental_id": range(1, 13),
    "station": ["A"] * 10 + ["B"] * 2,
    "day": [1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 1, 2],
    "customer": ["c1", "c1", "c2", "c1", "c1", "c2", "c3", "c1", "c1", "c4", "c2", "c5"],
    "duration_min": [10, 10, 12, 10, 10, 12, 90, 10, 10, 8, 12, 15],
})
print("12 rental events, 3 days, 2 stations, 5 customers")
events

## Four datasets, one set of events

Nothing below adds information. Every table is built from those twelve rows. What changes is **what
one row represents** - and with it, what question the table can answer.

In [ ]:
per_rental = events                                    # one row = one rental
per_station_day = (events.groupby(["station", "day"])
                   .agg(rentals=("rental_id", "size"),
                        mean_duration=("duration_min", "mean")).reset_index())
per_station = (events.groupby("station")
               .agg(rentals=("rental_id", "size"),
                    days_open=("day", "nunique")).reset_index())
per_customer = (events.groupby("customer")
                .agg(rentals=("rental_id", "size"),
                     mean_duration=("duration_min", "mean")).reset_index())

for name, table in [("per rental", per_rental), ("per station-day", per_station_day),
                    ("per station", per_station), ("per customer", per_customer)]:
    print(f"{name:<18} {len(table)} rows")

In [ ]:
print("ONE ROW = ONE STATION-DAY")
print(per_station_day.to_string(index=False))
print("\nONE ROW = ONE CUSTOMER")
print(per_customer.to_string(index=False))

### Each grain answers a different question

| One row is | Rows | It can answer | It cannot answer |
|---|---|---|---|
| **one rental** | 12 | How long is a typical *rental*? Which rentals are unusual? | How busy is a *day*? How loyal is a *customer*? |
| **one station-day** | 5 | How busy is a station on a given day? Should we forecast demand? | Anything about individual rentals or people |
| **one station** | 2 | How do the stations compare overall? | Anything that changes over time |
| **one customer** | 5 | Who are the heavy users? Will this person churn? | Anything about a particular day |

**The unit of observation is a decision, not a property of the data.** The same events support all
four, and the right choice comes from the *decision you are supporting*:

- Forecasting how many bikes to put out tomorrow -> **one row = one station-day**. The thing you
  predict is a day's demand, so a day must be a row.
- Deciding which customers to send an offer to -> **one row = one customer**. You act on people, so
  people must be rows.
- Detecting a suspicious rental -> **one row = one rental**.

**The test:** *what am I going to make a decision about?* That thing is your row. If you find
yourself unable to answer, you are not ready to model - and that is a finding, not a delay.

Two consequences that arrive immediately, and that later chapters are built on:

- **The target lives at the grain of the row.** "Rentals" means a count at the station-day grain and
  a lifetime total at the customer grain. Same word, different variable.
- **The split must respect the grain.** With one row per rental, customer c1's six rentals could
  land on both sides of a random split - the model would see the same person in training and test.
  That is the reason `GroupKFold` exists, and it is 04-04.

## The four kinds of column

Once you know what a row is, every column is one of four things. Sorting them takes a minute and
prevents a category of mistake.

| Kind | What it is | Example here | Danger |
|---|---|---|---|
| **Target** | What you want to predict | `rentals` at the station-day grain | Must be defined at the row's grain, and must not be knowable only after the fact |
| **Feature** | Information you will have when you predict | `station`, `day of week` | Only a feature if available *at prediction time* |
| **Identifier** | Names the row; carries no information | `rental_id`, `customer` | Feeding it to a model lets it memorise (00-01's rule C) |
| **Metadata** | Describes the record, not the thing | when the row was exported, source file | Often leaks - an export timestamp can encode the outcome |

The identifier row deserves emphasis. `rental_id` looks like a number and is not one: rental 12 is
not "larger" than rental 6, it is simply later. A model handed sequential ids will happily learn
that higher ids mean something, because in a growing dataset they usually correlate with time -
and then fails completely on new ids beyond the range it saw.

In [ ]:
column_roles = pd.DataFrame({
    "column": ["rental_id", "station", "day", "customer", "duration_min"],
    "kind": ["identifier", "feature", "feature", "identifier", "target or feature"],
    "why": ["names the row, carries no information",
            "known before the rental starts",
            "known in advance; day-of-week is the useful form",
            "names the person - but 'rentals so far' derived from it IS a feature",
            "depends on the question: predict it, or use it to describe past behaviour"],
})
print(column_roles.to_string(index=False))

Notice the last two rows. `customer` as a raw label is an identifier - useless and dangerous as a
model input. But *things derived from it* - how many rentals this customer has made **before this
one**, their average duration **so far** - are legitimate and often powerful features. The
identifier is the key you use to compute them, not the input itself.

And `duration_min` is a target in one project and a feature in another. Nothing about the column
says which. Only the question does.

## Units, and what arithmetic a column allows

Every column has a **measurement scale**, and the scale decides what you may legitimately compute.
This extends the test from 00-03.

| Scale | Test | Example | Legitimate |
|---|---|---|---|
| **Nominal** | Names with no order | `station`, blood type, postcode | Counts, modes. **Not** averages |
| **Ordinal** | Ordered, gaps not equal | ratings, pain 1-10, small/medium/large | Median, quantiles. Means only with a stated assumption |
| **Interval** | Equal gaps, no true zero | temperature in °C, calendar dates | Differences. **Ratios are meaningless**: 20 °C is not "twice as warm" as 10 °C |
| **Ratio** | Equal gaps and a true zero | `duration_min`, counts, money, distance | Everything, including ratios |

Two things follow, and both are mistakes people make constantly:

- **A postcode is not a number.** Nor is a customer id, a product code, or a route number. Their
  digits support no arithmetic at all. Feeding one to a model as a number teaches it that postcode
  50000 is "more" than 10000.
- **Averaging a satisfaction rating assumes the gap from 1 to 2 equals the gap from 4 to 5.** That
  is an assumption about people, not about data. It may be reasonable; say so rather than assuming
  it silently.

**And always know the unit.** `duration_min` is minutes. An error of 5 on it means five minutes. In
00-01 the whole point of MAE was that it carries the target's unit - a claim you can only make if
you know what that unit is.

### Predict before running

Maria asks: **"on an average day, how many rentals does a station get?"**

Two people compute it. One divides the total rentals by the number of station-days. The other takes
each station's average and averages those.

1. Will they get the same number?
2. If not, which is bigger, and why?
3. Which one answers Maria's question?

---

## Failure lab: two right answers to one question

In [ ]:
overall = per_station_day["rentals"].mean()
mean_of_means = per_station_day.groupby("station")["rentals"].mean().mean()

print("per-station averages:")
print(per_station_day.groupby("station")["rentals"].mean().round(3).to_string())
print()
print(f"total rentals / station-days  = 12 / 5 = {overall:.3f}")
print(f"average of the two station averages = {mean_of_means:.3f}")

### Diagnosis

**2.400 and 2.167.** Same twelve events, same five station-days, one question, two answers.

Neither calculation is wrong. They are **weighted differently**:

- `12 / 5 = 2.400` weights every **station-day** equally. Station A contributes three of the five
  days, so it counts three times.
- `(3.333 + 1.000) / 2 = 2.167` weights every **station** equally. Station A's busy days and
  station B's quiet ones count the same, even though A was open for three days and B for two.

That is the mean-of-means trap, and the rule behind it is worth memorising:

> **The average of averages is not the average, unless every group is the same size.**

Here A has 3 days and B has 2, so the two disagree. Make the groups equal and they coincide.

**Which answers Maria's question?** She asked "on an average day, how many rentals does a station
get?" - the row she has in mind is a station-day, so **2.400**. If she had asked "how does a typical
station perform?", the station is the row and 2.167 is right, because it refuses to let a station
that happened to be open longer dominate.

**Neither is a mistake. Computing one while meaning the other is.**

### The same trap, wearing a different hat

In [ ]:
per_rental_mean = events["duration_min"].mean()
per_customer_mean = per_customer["mean_duration"].mean()

print(per_customer.to_string(index=False))
print()
print(f"mean duration per RENTAL   = 209 / 12 = {per_rental_mean:.2f} minutes")
print(f"mean duration per CUSTOMER = 135 /  5 = {per_customer_mean:.2f} minutes")

**17.42 minutes and 27.00 minutes.** A gap of more than half, from the same twelve rentals.

Per rental, customer c1's six short trips dominate - they are half of all the rentals. Per customer,
c3's single 90-minute trip counts exactly as much as c1's entire history.

**Which is right depends on the decision:**

| The decision | The row | The number |
|---|---|---|
| Pricing per rental; sizing the bike fleet | one rental | **17.42 minutes** |
| Describing your customers; segmenting them | one customer | **27.00 minutes** |

And a third possibility nobody asked for but that is often the real question: **the median**. The
median rental is 10 minutes, because c3's 90-minute outlier moves the mean and not the median. If
the question is "what does a typical trip look like?", that gap matters more than the grain does.

### Remedies

| Remedy | What it catches | Cost |
|---|---|---|
| Write "one row = ..." at the top of every notebook | The whole class of error | One sentence |
| Say the unit out loud in every reported number | "2.4 rentals per station-day", not "average rentals" | Nothing, and it settles most arguments |
| Check whether the groups are equal-sized before averaging averages | The mean-of-means trap specifically | One `value_counts()` |
| Report the median alongside the mean | Skew being mistaken for level | One line |
| Ask "what am I deciding about?" before aggregating | Choosing the grain deliberately | A minute of thought |

The first one is the important one. **"One row = one station-day" written at the top of a notebook
is worth more than any amount of care applied afterwards**, because it lets a reader catch a
mismatch you cannot see yourself.

## Common misconceptions

**"The dataset determines what a row is."**
The *file* determines what a row is. The *project* determines what a row **should** be, and turning
one into the other is your job. Almost every real dataset arrives at the wrong grain.

**"Aggregating loses information, so finer is always better."**
Finer grain means more rows and more ways to leak. One row per rental with a random split puts the
same customer on both sides; one row per customer does not. The right grain is the one that matches
the decision, not the smallest one available.

**"An id column is harmless - the model will ignore it."**
It will not. A sequential id usually correlates with time, so the model learns from it and then
fails on ids outside the training range. Drop identifiers explicitly rather than hoping.

**"An average is an average."**
It is a weighted average, and the weights come from your choice of row. State the unit and the
ambiguity disappears.

**"Ordinal data can be averaged - everyone does it."**
Everyone does, and it assumes the gaps are equal. Sometimes defensible, never automatic. Say which
assumption you are making, or use the median.

**"Once I pick a grain I am stuck with it."**
You can and often should build several tables from the same events - one per customer for
segmentation, one per station-day for forecasting. What you must not do is compute a number at one
grain and describe it at another.

---

## Exercises

Solutions: `solutions/02_data_literacy/02-01_what_is_a_row_solutions.ipynb`.

### Quick understanding

**E1 (define).** In one sentence each: what is the unit of observation, and what are the four kinds
of column?

**E2 (explain).** Why is `customer` an identifier rather than a feature, and what *derived* from it
would be a legitimate feature?

**E3 (explain).** State the mean-of-means rule and the condition under which the two averages
agree.

### Hand calculation

**E4 (calculate).** Station A had 3, 4 and 3 rentals on its three days; station B had 1 and 1 on its
two. Compute by hand: rentals per station-day, and the average of the two station means. Then
suppose B had also been open on day 3 with 1 rental. Recompute both. Which number moved more, and
why?

**E5 (calculate).** From the twelve events, compute by hand: the mean duration per rental, the
median duration per rental, and the mean duration per customer. Explain in one sentence each why
the three differ.

### Coding

**E6 (code).** Build a fifth table: **one row per customer-day**. How many rows does it have? Give
one question it answers that none of the four tables in the chapter can.

**E7 (code).** Write `describe_grain(df, name)` printing: the number of rows, which columns are
unique per row (candidate identifiers), and which columns have exactly one value per row of the
grain. Run it on all four tables and say what it reveals about `per_station`.

### Interpretation

**E8 (interpret).** A report says "our average customer rents for 27 minutes". Give three questions
you would ask before quoting that number in a pricing decision.

### Debugging

**E9 (diagnose).** A colleague builds a churn model with one row per *rental* and a target of "did
this customer churn". Their held-out accuracy is 0.96 and the model fails in production. List three
things wrong with the setup, in order of severity, and name the one that explains the 0.96.

### Exam and interview reasoning

**E10 (defend).** *"What is the first question you ask on a new dataset?"* Answer in about 120
words, and say what you do when the answer is unclear.

**E11 (design).** A retailer wants to predict which products will run out of stock next week. Say
what one row should be, what the target is and its units, two features that would be available at
prediction time, and one column that looks like a feature and is not.

### Transfer to a different situation

**E12 (design).** A school has: one row per exam sitting, with student id, subject, date and mark.
Give the unit of observation, target and one difficulty for each of these questions:
(a) which students need extra support? (b) which subjects are getting harder? (c) will this student
pass this exam?

### Explain it to someone non-technical

**E13 (explain).** In under 70 words, explain to Maria why you got 2.4 and her assistant got 2.17,
without either of you making a mistake.

### Optional challenge

**E14 (code + diagnose).** Twelve rows are too few to show this, so generate a larger SYNTHETIC
set: 60 customers with between 3 and 20 rentals each, and a churn label assigned **per customer at
random** - deliberately unrelated to anything else, so the only way to "predict" it is to recognise
the person. Put one row per rental, encode `customer` as one-hot features, and fit a model twice:
once with a random split, once with a split that keeps each customer entirely on one side. Report
both accuracies, compare each against the majority-class baseline, and name the mechanism.

In [ ]:
# Your workspace. Still in memory: events, per_rental, per_station_day, per_station, per_customer.

## Mastery check

Without scrolling up, can you:

- [ ] Say what one row represents in each of the four tables, and one question each answers?
      *(If not: "Four datasets, one set of events".)*
- [ ] Name the four kinds of column and the danger of each? *(If not: "The four kinds of column".)*
- [ ] State the mean-of-means rule? *(If not: "Failure lab".)*
- [ ] Say why a postcode is not a number? *(If not: "Units, and what arithmetic".)*
- [ ] Explain why the grain determines how you must split the data? *(If not: "Each grain answers a
      different question".)*

## What should now feel instinctive

1. **"What does one row represent?"** - asked and *written down* before anything else.
2. **"What am I making a decision about?"** - that thing is your row.
3. **Every reported number carries its unit**: "2.4 rentals per station-day", never "average
   rentals".
4. **Identifiers are not features.** What you derive from them can be.
5. **An average of averages is a different number.** Check whether the groups are equal-sized.

## Flashcards

| Question | Answer |
|---|---|
| Unit of observation | What one row represents - a decision you make, not a property of the file |
| How do you choose it? | By what you will make a decision about |
| The four kinds of column | Target, feature, identifier, metadata |
| Why is an id dangerous as a feature? | Sequential ids correlate with time; the model learns them and fails outside the range |
| Can a customer id ever help? | Not raw. Things derived from it - rentals so far, average duration so far - are legitimate features |
| The mean-of-means rule | The average of averages is not the average unless every group is the same size |
| Why did 17.42 and 27.00 both come from 12 rentals? | One weights rentals equally, the other weights customers equally |
| Nominal, ordinal, interval, ratio | Names; ordered with unequal gaps; equal gaps without a true zero; equal gaps with a true zero |
| Why is 20 °C not twice 10 °C? | Celsius is an interval scale - its zero is arbitrary, so ratios are meaningless |
| How does grain affect splitting? | Rows about the same entity must not straddle a train/test split - hence grouped splits (04-04) |

## Next

**02-02 · Where data comes from: provenance and the collection process.**

You now know how to ask what a row is. The next question is harder and less often asked: **how did
this row come to exist?** Somebody or something decided to record it, at a moment, by a method, with
a purpose that was probably not yours. Every gap, every sentinel value, every category that means
two things has a reason - and the reason is usually visible only in the collection process, not in
the file.

New terms are in [GLOSSARY.md](../../GLOSSARY.md).